# Task 3 — CPU preparation

Choose Runtime → Change runtime type → Hardware accelerator **None**. Run both cells. Upload `ATML_PA2_Task3_Colab.zip` when prompted. This downloads pinned course assets and model files to Drive, runs CPU tests, freezes prompt/seed schedules and computes the supplied-cache study. It does not load policy weights or train. An existing experiment requires an explicit same-contract resume.

No reported result exists until its job actually runs. The standard endpoint is reserved for Task 4. New experimental choices require explicit approval.

In [ ]:
import time
SESSION_START_EPOCH = time.time()
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path
from google.colab import files, drive
uploaded = files.upload()  # Upload ATML_PA2_Task3_Colab.zip.
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly the Task 3 package ZIP.'
assert Path(archives[0]).name == archives[0], 'Upload a filename without directories.'
archive_path = Path('/content') / archives[0]
archive_path.write_bytes(uploaded[archives[0]])
del uploaded
package = Path('/content/task3_package')
package.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as z:
    names = z.namelist()
    assert len(names) == len(set(names)), 'Duplicate archive members.'
    manifest = json.loads(z.read('PACKAGE_SHA256.json'))
    assert set(names) == set(manifest) | {'PACKAGE_SHA256.json'}, 'Package file set differs.'
    for item in z.infolist():
        target = (package / item.filename).resolve()
        assert package.resolve() in target.parents, 'Unsafe archive path.'
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Archive contains a link.'
    for name,digest in manifest.items():
        payload = z.read(name)
        assert hashlib.sha256(payload).hexdigest() == digest, 'Package checksum differs: ' + name
        target = package / name
        target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():
            assert target.read_bytes() == payload, 'Existing package differs; start a fresh Colab runtime.'
        else:
            target.write_bytes(payload)
expected_source = 'f87021b8ee1057d2afbae29d12c5bd0e904d10b1e9888d9476f99f2959a05c07'
assert hashlib.sha256((package/'task3/source/docs/SOURCE_SHA256.json').read_bytes()).hexdigest() == expected_source
runtime = Path('/content/task3_runtime/course')
subprocess.run([sys.executable,str(package/'task3/restore_source.py'),'--destination',str(runtime)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(runtime/'requirements.txt')],check=True)
drive.mount('/content/drive')
EXPERIMENT = '/content/drive/MyDrive/ATML-Assignment-2/task3/experiment_v1'
os.chdir(runtime)
print('Sealed Task 3 source ready. Scientific settings are fixed in the approved package.')

In [ ]:
# First preparation: False. Set True only to resume this SAME package/contract.
RESUME_PREPARATION = False
command = [sys.executable,'-m','task3_grpo.prepare','--experiment',EXPERIMENT]
if RESUME_PREPARATION:
    command.append('--resume')
subprocess.run(command,check=True)
receipt = json.loads((Path(EXPERIMENT)/'PREPARATION.json').read_text())
assert receipt['full_course_asset_validator_passed'] and not receipt['policy_training_performed']
print('CPU preparation verified on Drive. Next open the GPU notebook and choose a GPU runtime.')
drive.flush_and_unmount()